# 🤖 Sales Amount Prediction — Advanced Machine Learning

## Project Overview
This notebook extends the Sales Data Analysis project into a complete Machine Learning workflow.

### Objective
Predict **Sales Amount (`Amount`)** from customer, demographic, geographic, sector, product-category, and order-related features.

### ML Workflow
1. Load and inspect the dataset
2. Clean and validate data
3. Explore the target variable
4. Select features and target
5. Build numerical/categorical preprocessing pipelines
6. Split data into training and testing sets
7. Train multiple regression models
8. Evaluate models using MAE, RMSE and R²
9. Tune the strongest tree-based model
10. Compare baseline vs tuned models
11. Analyze feature importance
12. Visualize actual vs predicted values and residuals
13. Save the best trained pipeline
14. Summarize final business/ML findings

> **Target variable:** `Amount`


# 📚 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_FILE = "project Data new.xlsx"

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

# 📂 2. Load Dataset

In [ ]:
path = Path(DATA_FILE)

if not path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}. "
        "Place the Excel file in the same folder as this notebook."
    )

df = pd.read_excel(path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

# 🔍 3. Dataset Inspection

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("Missing Count"))

print("\nDuplicate Rows:", df.duplicated().sum())

# 🧹 4. Data Cleaning & Validation

The cleaning follows the existing Sales ML project:
- Rename `Occupation` to `Sector`
- Convert `Marital_Status` values `1` → `Married` and `0` → `Single`
- Remove `unnamed1` and `Status` when present
- Remove rows where the target `Amount` is missing
- Remove duplicate rows


In [ ]:
df = df.copy()

# Rename Occupation to Sector
if "Occupation" in df.columns and "Sector" not in df.columns:
    df.rename(columns={"Occupation": "Sector"}, inplace=True)

# Convert marital status values
if "Marital_Status" in df.columns:
    df["Marital_Status"] = (
        df["Marital_Status"]
        .astype(str)
        .replace({"1": "Married", "0": "Single"})
    )

# Remove unnecessary columns when available
columns_to_drop = [c for c in ["unnamed1", "Status"] if c in df.columns]
if columns_to_drop:
    df.drop(columns=columns_to_drop, inplace=True)

# Target must exist
if "Amount" not in df.columns:
    raise ValueError("Target column 'Amount' was not found.")

# Remove missing target rows and duplicates
df = df.dropna(subset=["Amount"]).drop_duplicates().copy()

print("Cleaned Dataset Shape:", df.shape)
display(df.head())

# 📊 5. Target Variable Analysis

In [ ]:
print("Target Summary:")
display(df["Amount"].describe().to_frame("Amount"))

fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df["Amount"], bins=50, kde=True, ax=ax)
ax.set_title("Distribution of Sales Amount")
ax.set_xlabel("Sales Amount (₹)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Target Skewness:", round(df["Amount"].skew(), 3))

# 🎯 6. Feature Selection

The original Sales ML project uses these predictive features:

- Gender
- Age Group
- Age
- Marital Status
- State
- Zone
- Sector
- Product Category
- Orders

Identifier/name columns such as `User_ID`, `Cust_name`, and `Product_ID` are intentionally excluded.


In [ ]:
candidate_features = [
    "Gender",
    "Age Group",
    "Age",
    "Marital_Status",
    "State",
    "Zone",
    "Sector",
    "Product_Category",
    "Orders"
]

features = [c for c in candidate_features if c in df.columns]
missing_features = [c for c in candidate_features if c not in df.columns]

if missing_features:
    print("Warning — missing candidate features:", missing_features)

if not features:
    raise ValueError("No valid ML features were found.")

target = "Amount"

X = df[features].copy()
y = df[target].copy()

print("Features used:")
print(features)
print("\nTarget:", target)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

# ✂️ 7. Train-Test Split

- 80% training data
- 20% testing data
- `random_state=42` for reproducibility


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# 🔤 8. Identify Numerical & Categorical Features

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical Features:", categorical_features)
print("Numerical Features:", numerical_features)

# ⚙️ 9. Build Preprocessing Pipelines

Categorical columns are:
- imputed using the most frequent value
- encoded with One-Hot Encoding

Numerical columns are:
- imputed using the median
- scaled for the Linear Regression baseline


In [ ]:
numeric_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_scaled, numerical_features),
        ("cat", categorical, categorical_features)
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_tree, numerical_features),
        ("cat", categorical, categorical_features)
    ]
)

print("Preprocessing pipelines created.")

# 📏 10. Evaluation Function

In [ ]:
def evaluate_model(model_name, model, X_eval, y_eval):
    predictions = model.predict(X_eval)

    mae = mean_absolute_error(y_eval, predictions)
    rmse = np.sqrt(mean_squared_error(y_eval, predictions))
    r2 = r2_score(y_eval, predictions)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    }

# 📈 11. Baseline — Linear Regression

In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor_linear),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)

linear_result = evaluate_model(
    "Linear Regression",
    linear_model,
    X_test,
    y_test
)

pd.DataFrame([linear_result])

# 🌲 12. Random Forest Regression

In [ ]:
random_forest = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", RandomForestRegressor(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_features="sqrt"
    ))
])

random_forest.fit(X_train, y_train)

rf_result = evaluate_model(
    "Random Forest",
    random_forest,
    X_test,
    y_test
)

pd.DataFrame([rf_result])

# 🌳 13. Extra Trees Regression

In [ ]:
extra_trees = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", ExtraTreesRegressor(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        max_features=1.0
    ))
])

extra_trees.fit(X_train, y_train)

extra_result = evaluate_model(
    "Extra Trees",
    extra_trees,
    X_test,
    y_test
)

pd.DataFrame([extra_result])

# 🚀 14. Gradient Boosting Regression

Gradient Boosting is included as another non-linear model for comparison.


In [ ]:
gradient_boosting = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", GradientBoostingRegressor(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE
    ))
])

gradient_boosting.fit(X_train, y_train)

gb_result = evaluate_model(
    "Gradient Boosting",
    gradient_boosting,
    X_test,
    y_test
)

pd.DataFrame([gb_result])

# 🏆 15. Baseline Model Comparison

In [ ]:
baseline_results = pd.DataFrame([
    linear_result,
    rf_result,
    extra_result,
    gb_result
]).sort_values("R2 Score", ascending=False).reset_index(drop=True)

display(baseline_results)

best_baseline_name = baseline_results.loc[0, "Model"]
print("Best baseline model:", best_baseline_name)

# 🎯 16. Hyperparameter Tuning — Extra Trees

The strongest baseline tree model is tuned using `RandomizedSearchCV`.

The search optimizes **R²** on the training set using 5-fold cross-validation.


In [ ]:
extra_trees_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("model", ExtraTreesRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

param_distributions = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [None, 10, 15, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.5, 0.75, 1.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=extra_trees_pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="r2",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

print("Best CV R²:", round(search.best_score_, 4))
print("\nBest Parameters:")
print(search.best_params_)

# 📊 17. Tuned Model Evaluation

In [ ]:
tuned_result = evaluate_model(
    "Tuned Extra Trees",
    search.best_estimator_,
    X_test,
    y_test
)

all_results = pd.concat(
    [baseline_results, pd.DataFrame([tuned_result])],
    ignore_index=True
).sort_values("R2 Score", ascending=False).reset_index(drop=True)

display(all_results)

best_model_name = all_results.loc[0, "Model"]
print("🏆 Best final model:", best_model_name)

# 📉 18. Model Performance Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=all_results,
    x="Model",
    y="R2 Score",
    ax=ax
)
ax.set_title("Model Comparison — R² Score")
ax.set_xlabel("Model")
ax.set_ylabel("R² Score")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

display(all_results.style.format({
    "MAE": "₹{:,.2f}",
    "RMSE": "₹{:,.2f}",
    "R2 Score": "{:.4f}"
}))

# 🔮 19. Actual vs Predicted — Best Model

In [ ]:
model_lookup = {
    "Linear Regression": linear_model,
    "Random Forest": random_forest,
    "Extra Trees": extra_trees,
    "Gradient Boosting": gradient_boosting,
    "Tuned Extra Trees": search.best_estimator_
}

best_model = model_lookup[best_model_name]
best_predictions = best_model.predict(X_test)

comparison = pd.DataFrame({
    "Actual Amount": y_test.values,
    "Predicted Amount": best_predictions
})

comparison["Error"] = (
    comparison["Actual Amount"] - comparison["Predicted Amount"]
)
comparison["Absolute Error"] = comparison["Error"].abs()

display(comparison.head(10))

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    comparison["Actual Amount"],
    comparison["Predicted Amount"],
    alpha=0.5
)

min_value = min(comparison["Actual Amount"].min(), comparison["Predicted Amount"].min())
max_value = max(comparison["Actual Amount"].max(), comparison["Predicted Amount"].max())

ax.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

ax.set_title(f"Actual vs Predicted Amount — {best_model_name}")
ax.set_xlabel("Actual Amount (₹)")
ax.set_ylabel("Predicted Amount (₹)")
ax.grid()
plt.tight_layout()
plt.show()

# 🔎 20. Prediction Error Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(
    comparison["Error"],
    bins=40,
    kde=True,
    ax=ax
)
ax.axvline(0, linestyle="--")
ax.set_title("Distribution of Prediction Errors")
ax.set_xlabel("Prediction Error (₹)")
ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

print("Mean Absolute Error:", f"₹{comparison['Absolute Error'].mean():,.2f}")
print("Median Absolute Error:", f"₹{comparison['Absolute Error'].median():,.2f}")

# ⭐ 21. Feature Importance

For the final tree-based model, feature importance is extracted from the trained preprocessing + model pipeline.

The chart shows which encoded features contribute most strongly to the model.


In [ ]:
final_pipeline = best_model
preprocessor_fitted = final_pipeline.named_steps["preprocessor"]
tree_model = final_pipeline.named_steps["model"]

if hasattr(tree_model, "feature_importances_"):
    feature_names = preprocessor_fitted.get_feature_names_out()
    importances = tree_model.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(20))

    fig, ax = plt.subplots(figsize=(10, 7))
    top_features = importance_df.head(20).sort_values("Importance")
    ax.barh(top_features["Feature"], top_features["Importance"])
    ax.set_title("Top 20 Feature Importances")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Feature importance is not available for the selected final model.")

# 🧪 22. Sample Prediction Function

This helper lets you pass a single row of feature values through the same preprocessing and final model used during training.


In [ ]:
def predict_sales_amount(input_data):
    """Predict sales Amount for one or more records.

    input_data must contain the feature columns used by the model.
    """
    input_df = pd.DataFrame(input_data)
    missing = [c for c in features if c not in input_df.columns]

    if missing:
        raise ValueError(f"Missing required features: {missing}")

    predictions = best_model.predict(input_df[features])
    return pd.DataFrame({
        "Predicted Amount": predictions
    })

# Example using real test rows
sample_input = X_test.head(3).copy()
sample_predictions = predict_sales_amount(sample_input)

display(sample_predictions)

# 💾 23. Save the Best Model

The complete preprocessing + model pipeline is saved together so future predictions use the exact same transformations.


In [ ]:
import joblib

MODEL_FILE = "sales_amount_best_model.pkl"

joblib.dump(best_model, MODEL_FILE)

print(f"Saved best model pipeline: {MODEL_FILE}")

# 📝 24. Final Model Metrics

In [ ]:
final_metrics = {
    "Best Model": best_model_name,
    "MAE": tuned_result["MAE"] if best_model_name == "Tuned Extra Trees" else all_results.loc[0, "MAE"],
    "RMSE": tuned_result["RMSE"] if best_model_name == "Tuned Extra Trees" else all_results.loc[0, "RMSE"],
    "R² Score": tuned_result["R2 Score"] if best_model_name == "Tuned Extra Trees" else all_results.loc[0, "R2 Score"]
}

display(pd.DataFrame([final_metrics]).style.format({
    "MAE": "₹{:,.2f}",
    "RMSE": "₹{:,.2f}",
    "R² Score": "{:.4f}"
}))

# 🏁 25. Conclusion

This project demonstrates an end-to-end Machine Learning regression workflow for predicting sales amount.

### Covered Skills
- Python
- Pandas & NumPy
- Data cleaning
- Exploratory target analysis
- Feature engineering / feature selection
- One-Hot Encoding
- Scikit-learn Pipelines
- Train/Test Split
- Linear Regression
- Random Forest
- Extra Trees
- Gradient Boosting
- Hyperparameter tuning with RandomizedSearchCV
- Cross-validation
- MAE, RMSE and R²
- Residual/error analysis
- Feature importance
- Model persistence with Joblib

### Important Modeling Note
`Orders` is included because it was part of the original Sales ML project. If the real business use case is to predict the amount **before an order quantity is known**, retrain a second version without `Orders` to avoid using information that may not be available at prediction time.

> Run all cells from top to bottom to generate the actual metrics and select the best model for the dataset.
